# LFPs plots - Granger Causality

Required files:
- TTL times
- TTL labels
- LFPs data (can be read using TTL times)
- Sleep/Wake labels


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
import glob

import os
import re

def find_files(root_folder, pattern):
    # Compile the regex pattern for matching
    regex = re.compile(pattern)
    
    matching_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for filename in filenames:
            if regex.match(filename):
                file_path = os.path.join(dirpath, filename)
                matching_files.append(file_path)
                # print(f"File found: {file_path}")
    return matching_files

MOUSE_NAME = 'C9'
main_path = r'\\path\Pixel1\1_auditory_neuropixels_BarakH'
main_folders = glob.glob(main_path + f'\*12_{MOUSE_NAME}_*')
SleepWakeLabels_fname = 'labels_All.mat'
TTL_times_fname = 'TTL_times_adj.txt'
TTL_labels_fname = 'TTL_labels.txt'
LFP_path_fname = r'.*\.lf\.bin$'

## Collect LFPs per SoundType, All Channels

In [ ]:
import pandas as pd
from Global_functions import read_file_by_time_steps
from pathlib import Path

# SoundType = [101, 102, 400, 1301, 1401, 1402, 1403]
SoundType = [101]
chanList = list(range(1, 380)) # List of channels to plot

main_folder = main_folders[0]    
cur_date = os.path.splitext(os.path.basename(main_folder))[0].split('_')[0]

LFP_path = find_files(main_folder, LFP_path_fname)[0]
TTL_labels_path = find_files(main_folder, TTL_labels_fname)[0]
TTL_times_path = find_files(main_folder, TTL_times_fname)[0]
SleepWakeLabels_path = find_files(main_folder, SleepWakeLabels_fname)[0]

TTL_times = np.loadtxt(TTL_times_path)
TTL_labels = np.loadtxt(TTL_labels_path)
SleepWakeLabels =  scipy.io.loadmat(SleepWakeLabels_path)
SleepWakeLabels = np.squeeze(SleepWakeLabels['labels'])
SleepWakeTimes = np.linspace(0, 2*60*60, len(SleepWakeLabels))  # the stop value is set by the time I used for manual scoring
print(f'\033[91m Date \033[0m{cur_date}')

''' Pick SoundType '''
if SoundType != 400:
    num_trials = np.sum(TTL_labels == SoundType)
    cur_SoundType_trialsTimes = TTL_times[TTL_labels == SoundType]
else:
    num_trials = np.sum(np.bitwise_and(TTL_labels >= 400,TTL_labels <= 600))
    cur_SoundType_trialsTimes = TTL_times[np.bitwise_and(TTL_labels >= 400,TTL_labels <= 600)]

print(f'\033[94m SoundType \033[0m{SoundType} - \033[96m Number of trials: \033[0m{num_trials}')
cur_SoundType_trialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes > 1]

''' Find Sleep/Wake label '''
closest_SleepWakeTimes = []
for curTTLtime in cur_SoundType_trialsTimes:
    closest_SleepWakeTimes.append(np.argmin(np.abs(SleepWakeTimes - curTTLtime)))

cur_SoundType_SleepWakeLabels = SleepWakeLabels[closest_SleepWakeTimes]

''' Extract Trial times '''
All_TrialsTimes = cur_SoundType_trialsTimes
First30min_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes<30*60]
Last30min_TrialsTimes = cur_SoundType_trialsTimes[3*30*60<cur_SoundType_trialsTimes]
Sleep_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 3]
Wake_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 2]
print(f'Sleep/Wake/1st/last - {len(Sleep_TrialsTimes)}/{len(Wake_TrialsTimes)}/{len(First30min_TrialsTimes)}/{len(Last30min_TrialsTimes)}')

''' Read LFPs at channels All Trials '''
Picked_TrialsTimes = All_TrialsTimes
Trials_read_times_start = Picked_TrialsTimes - 1 # Take one second before Stimuli Onset
tstep = 4.5 # Take 4.5 seconds after -1 from onset (3.5 seconds after stimuli)

LFPs_by_trials, sr_LFP = read_file_by_time_steps(Path(LFP_path), Trials_read_times_start, tstep, chanList)
min_length = min(arr.shape[1] for arr in LFPs_by_trials)
LFPs_by_trials = [arr[:, :min_length] for arr in LFPs_by_trials]
LFPs_by_trials = np.stack(LFPs_by_trials, axis=1)  # 'LFPs_by_trials' np array with shape (N_channels, N_trials, len_Trial)

t_LFP = np.linspace(-1, tstep - 1, LFPs_by_trials.shape[2])
Trials_label = list(range(0,len(Picked_TrialsTimes)))
    
Trials_sleep = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Sleep_TrialsTimes])
Trials_Wake = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Wake_TrialsTimes])
Trials_first30min = np.array([np.where(Picked_TrialsTimes == j)[0] for j in First30min_TrialsTimes])
Trials_last30min = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Last30min_TrialsTimes])

In [ ]:
LFPs_by_trials_Sleep = np.squeeze(LFPs_by_trials[:, Trials_sleep, :])
LFPs_by_trials_Wake = np.squeeze(LFPs_by_trials[:, Trials_Wake, :])
LFPs_Sleep_by_channel = np.squeeze(np.mean(LFPs_by_trials_Sleep, axis=1))
LFPs_Wake_by_channel = np.squeeze(np.mean(LFPs_by_trials_Wake, axis=1))

# Granger Causality

In [ ]:
%matplotlib qt

import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests

def grangers_causation_matrix(data, variables, maxlag, test='ssr_chi2test', verbose=False):    
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r, c]], maxlag=maxlag)
            p_values = [round(test_result[i+1][0][test][1],6) for i in range(maxlag)]
            if verbose: print(f'Y = {r}, X = {c}, P Values = {p_values}')
            min_p_value = np.min(p_values)
            lag_min = None
            if min_p_value < 0.05:
                lag_min = 1/ (np.argmin(p_values) + 1)
            df.loc[r, c] = lag_min
    df.columns = [var + '_x' for var in variables]
    df.index = [var + '_y' for var in variables]
    return df

ds_step = 20
t_sample = t_LFP[::ds_step]
sr_sample = 1 / (t_sample[1] - t_sample[0])
lag = int(0.02 * sr_sample)

LFPs_Sleep_by_channel_norm = LFPs_Sleep_by_channel[:,::ds_step] - np.mean(LFPs_Sleep_by_channel[:,::ds_step], axis=0)
df_data = pd.DataFrame(LFPs_Sleep_by_channel_norm)

# df_data = pd.DataFrame({'140': sample_data, '130': shifted_sample_data, '150': shifted_sample_data1})
display(df_data)

df_gc = grangers_causation_matrix(df_data, df_data.columns, 5 * lag, test='ssr_chi2test', verbose=True)
display(df_gc)

plt.imshow(df_gc)
